In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/MachineLearningRating_v3.txt', sep='|')
# Create target
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

# Convert to numeric
df['TotalClaims'] = pd.to_numeric(df['TotalClaims'], errors='coerce')
df['TotalPremium'] = pd.to_numeric(df['TotalPremium'], errors='coerce')

# Select columns
selected_columns = [
    'TotalClaims', 'TotalPremium', 'HasClaim',
    'Gender', 'Province', 'MaritalStatus', 'Title',
    'VehicleType', 'make', 'Model', 'RegistrationYear', 
    'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors',
    'CoverType', 'Product', 'SumInsured', 'ExcessSelected',
    'AlarmImmobiliser', 'TrackingDevice'
]

existing_columns = [col for col in selected_columns if col in df.columns]
df_clean = df[existing_columns].copy()

print("Identify numeric vs categorical columns")

# Define which columns are truly numeric
numeric_cols_true = ['RegistrationYear', 'Cylinders', 'cubiccapacity', 
                     'kilowatts', 'NumberOfDoors', 'SumInsured', 'TotalClaims', 'TotalPremium']

# Define which columns are categorical (even if they contain numbers)
categorical_cols_true = ['Gender', 'Province', 'MaritalStatus', 'Title', 'VehicleType', 
                         'make', 'Model', 'bodytype', 'CoverType', 'Product', 
                         'ExcessSelected', 'AlarmImmobiliser', 'TrackingDevice']

print(f"Numeric columns: {numeric_cols_true}")
print(len(numeric_cols_true))
print(f"Categorical columns: {categorical_cols_true}")
print(len(categorical_cols_true))

Identify numeric vs categorical columns
Numeric columns: ['RegistrationYear', 'Cylinders', 'cubiccapacity', 'kilowatts', 'NumberOfDoors', 'SumInsured', 'TotalClaims', 'TotalPremium']
8
Categorical columns: ['Gender', 'Province', 'MaritalStatus', 'Title', 'VehicleType', 'make', 'Model', 'bodytype', 'CoverType', 'Product', 'ExcessSelected', 'AlarmImmobiliser', 'TrackingDevice']
13


In [18]:
df.columns

Index(['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth',
       'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language',
       'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province',
       'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode',
       'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders',
       'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors',
       'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser',
       'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff',
       'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet',
       'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm',
       'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section',
       'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium',
       'TotalClaims', 'HasClaim'],
      dtype='str')

In [28]:
#Fill missing values - NUMERIC columns"

for col in numeric_cols_true:
    if col in df_clean.columns:
        missing_count = df_clean[col].isnull().sum()
        if missing_count > 0:
            median_val = df_clean[col].median()
            df_clean.loc[df_clean[col].isnull(), col] = median_val
            print(f"Filled {col}: {missing_count:,} missing values with median: {median_val}")
        else:
            print(f"{col}: no missing values")

RegistrationYear: no missing values
Cylinders: no missing values
cubiccapacity: no missing values
kilowatts: no missing values
NumberOfDoors: no missing values
SumInsured: no missing values
TotalClaims: no missing values
TotalPremium: no missing values


In [29]:
#Fill remaining CATEGORICAL columns
# List of categorical columns that still have missing values
remaining_categorical = ['Gender', 'MaritalStatus', 'VehicleType', 'make', 'Model', 'bodytype']

for col in remaining_categorical:
    if col in df_clean.columns:
        # Check current missing count
        missing_before = df_clean[col].isnull().sum()
        print(f"\n{col}: {missing_before:,} missing values before")
        
        # Convert to string first
        df_clean[col] = df_clean[col].astype(str)
        
        # Replace 'nan' string with 'Unknown'
        df_clean.loc[df_clean[col] == 'nan', col] = 'Unknown'
        
        # Also fill any actual NaN values
        df_clean.loc[df_clean[col].isnull(), col] = 'Unknown'
        
        # Verify
        missing_after = (df_clean[col] == 'Unknown').sum()
        print(f"  ✓ Filled: {missing_after:,} values set to 'Unknown'")


Gender: 9,536 missing values before
  ✓ Filled: 9,536 values set to 'Unknown'

MaritalStatus: 8,259 missing values before
  ✓ Filled: 8,259 values set to 'Unknown'

VehicleType: 552 missing values before
  ✓ Filled: 552 values set to 'Unknown'

make: 552 missing values before
  ✓ Filled: 552 values set to 'Unknown'

Model: 552 missing values before
  ✓ Filled: 552 values set to 'Unknown'

bodytype: 552 missing values before
  ✓ Filled: 552 values set to 'Unknown'


In [30]:
#Final Verification"
# Check all columns for missing values
missing_check = df_clean.isnull().sum()
missing_cols = missing_check[missing_check > 0]

if len(missing_cols) == 0:
    print("No missing values in any column!")
else:
    print(f"Still have missing values in: {missing_cols.index.tolist()}")
    print(missing_cols)

# Also check for 'nan' strings
print("Checking for 'nan' strings:")
for col in df_clean.select_dtypes(include=['object']).columns:
    nan_count = (df_clean[col] == 'nan').sum()
    if nan_count > 0:
        print(f"{col}: {nan_count} 'nan' strings found - fixing...")
        df_clean.loc[df_clean[col] == 'nan', col] = 'Unknown'
    else:
        print(f"{col}: clean")

print(f"\nFinal shape: {df_clean.shape}")
print(f"Total missing values: {df_clean.isnull().sum().sum()}")

No missing values in any column!
Checking for 'nan' strings:
Gender: clean
Province: clean
MaritalStatus: clean
Title: clean
VehicleType: clean
make: clean
Model: clean
bodytype: clean
CoverType: clean
Product: clean
ExcessSelected: clean
AlarmImmobiliser: clean
TrackingDevice: clean

Final shape: (1000098, 22)
Total missing values: 0


# Data Preparation & Cleaning Summary

## Overview

The raw dataset `MachineLearningRating_v3.txt` contained **1,000,098 rows** and **52 columns**. Significant data cleaning was required to prepare the data for predictive modeling. This document outlines the cleaning steps performed.

---

## Initial Data Assessment

| Metric | Value |
|--------|-------|
| Total rows | 1,000,098 |
| Total columns | 52 |
| File format | Pipe-delimited (`|`) |
| Time period | Feb 2014 – Aug 2015 |

### Column Types Identified

| Type | Count | Examples |
|------|-------|----------|
| Numeric | 15 | TotalPremium, TotalClaims, RegistrationYear, Cylinders |
| Categorical | 35 | Gender, Province, VehicleType, make, Model, CoverType |
| Boolean | 1 | IsVATRegistered |
| Date | 1 | TransactionMonth |

---

## Missing Value Analysis

### Columns with >50% Missing Values (Dropped)

These columns were **removed** because they lacked sufficient data for meaningful analysis:

| Column | Missing % | Reason for Dropping |
|--------|-----------|---------------------|
| CrossBorder | 99.93% | Almost all values missing |
| NumberOfVehiclesInFleet | 100% | Completely empty |
| WrittenOff | 64.18% | Majority missing |
| Rebuilt | 64.18% | Majority missing |
| Converted | 64.18% | Majority missing |
| CustomValueEstimate | 77.96% | Majority missing |

### Columns with <50% Missing Values (Kept & Filled)

These columns were **retained** and missing values were imputed:

| Column | Missing Count | Missing % | Handling Method |
|--------|---------------|-----------|-----------------|
| Bank | 145,961 | 14.59% | Dropped (not needed for modeling) |
| AccountType | 40,232 | 4.02% | Dropped (not needed for modeling) |
| Gender | 9,536 | 0.95% | Filled with 'Unknown' |
| MaritalStatus | 8,259 | 0.83% | Filled with 'Unknown' |
| VehicleType | 552 | 0.06% | Filled with 'Unknown' |
| make | 552 | 0.06% | Filled with 'Unknown' |
| Model | 552 | 0.06% | Filled with 'Unknown' |
| bodytype | 552 | 0.06% | Filled with 'Unknown' |
| Cylinders | 552 | 0.06% | Filled with median (4.0) |
| cubiccapacity | 552 | 0.06% | Filled with median (2694.0) |
| kilowatts | 552 | 0.06% | Filled with median (111.0) |
| NumberOfDoors | 552 | 0.06% | Filled with median (4.0) |
| NewVehicle | 153,295 | 15.33% | Dropped (not needed for modeling) |

---

## Columns Selected for Modeling

After cleaning, **22 columns** were retained for modeling:

### Target Variables

| Column | Description | Type |
|--------|-------------|------|
| `TotalClaims` | Claim amount (Rands) | Numeric |
| `TotalPremium` | Premium paid (Rands) | Numeric |
| `HasClaim` | Binary indicator (1 = claim occurred) | Binary |

### Feature Variables

| Category | Columns |
|----------|---------|
| **Demographics** | Gender, Province, MaritalStatus, Title |
| **Vehicle Information** | VehicleType, make, Model, RegistrationYear, Cylinders, cubiccapacity, kilowatts, bodytype, NumberOfDoors |
| **Policy Information** | CoverType, Product, SumInsured, ExcessSelected |
| **Safety Features** | AlarmImmobiliser, TrackingDevice |

---

## Missing Value Imputation Strategy

### Numeric Columns

```python
# Filled with median to avoid skewing distributions
df_clean['Cylinders'].fillna(df_clean['Cylinders'].median(), inplace=True)
df_clean['cubiccapacity'].fillna(df_clean['cubiccapacity'].median(), inplace=True)
df_clean['kilowatts'].fillna(df_clean['kilowatts'].median(), inplace=True)
df_clean['NumberOfDoors'].fillna(df_clean['NumberOfDoors'].median(), inplace=True)
```

In [31]:
#FEATURE ENGINEERING - COMPLETE"
# Make a copy of clean data
df_model = df_clean.copy()
# 1. VEHICLE AGE FEATURES
# ============================================
print("1. VEHICLE AGE FEATURES")

# Vehicle age in years
current_year = 2015
df_model['VehicleAge'] = current_year - df_model['RegistrationYear']
df_model['VehicleAge'] = df_model['VehicleAge'].clip(0, 30)  # Cap at 30 years
print(" Created 'VehicleAge' (years)")

# Vehicle age category
df_model['VehicleAgeCat'] = pd.cut(df_model['VehicleAge'], 
                                    bins=[-1, 2, 5, 10, 30],
                                    labels=['0-2 years (New)', '3-5 years (Like New)', 
                                           '6-10 years (Used)', '10+ years (Old)'])
print("Created 'VehicleAgeCat' (New/Like New/Used/Old)")

# Is vehicle new? (less than 2 years old)
df_model['IsNewVehicle'] = (df_model['VehicleAge'] <= 2).astype(int)
print("Created 'IsNewVehicle' (binary)")

# 2. ENGINE PERFORMANCE FEATURES
print("ENGINE PERFORMANCE FEATURES")

# Power to weight ratio (kilowatts per cubic capacity)
df_model['PowerToWeight'] = df_model['kilowatts'] / (df_model['cubiccapacity'] + 1)
df_model['PowerToWeight'] = df_model['PowerToWeight'].round(3)
print("Created 'PowerToWeight' (kW/cc)")

# Engine size category
df_model['EngineSizeCat'] = pd.cut(df_model['cubiccapacity'],
                                    bins=[0, 1400, 1800, 2200, 10000],
                                    labels=['Small (<1.4L)', 'Medium (1.4-1.8L)', 
                                           'Large (1.8-2.2L)', 'Extra Large (>2.2L)'])
print("Created 'EngineSizeCat'")

# Power category
df_model['PowerCat'] = pd.cut(df_model['kilowatts'],
                               bins=[0, 100, 200, 300, 1000],
                               labels=['Low (<100kW)', 'Medium (100-200kW)', 
                                      'High (200-300kW)', 'Very High (>300kW)'])
print("Created 'PowerCat'")

# 3. PREMIUM-BASED FEATURES
print("3. PREMIUM-BASED FEATURES")

# Premium per cubic capacity
df_model['PremiumPerCC'] = df_model['TotalPremium'] / (df_model['cubiccapacity'] + 1)
df_model['PremiumPerCC'] = df_model['PremiumPerCC'].round(2)
print("Created 'PremiumPerCC' (R per cc)")

# Premium category (based on distribution)
df_model['PremiumCat'] = pd.qcut(df_model['TotalPremium'], 
                                  q=4, 
                                  labels=['Low Premium', 'Medium Premium', 
                                         'High Premium', 'Very High Premium'])
print("Created 'PremiumCat' (quartiles)")

# 4. SAFETY FEATURES

print("4. SAFETY FEATURES")

# Has alarm (binary)
if 'AlarmImmobiliser' in df_model.columns:
    df_model['HasAlarm'] = (df_model['AlarmImmobiliser'] == 'Yes').astype(int)
    print("Created 'HasAlarm' (binary)")

# Has tracking device (binary)
if 'TrackingDevice' in df_model.columns:
    df_model['HasTracking'] = (df_model['TrackingDevice'] == 'Yes').astype(int)
    print("Created 'HasTracking' (binary)")

# Safety score (alarm + tracking)
df_model['SafetyScore'] = df_model.get('HasAlarm', 0) + df_model.get('HasTracking', 0)
print("Created 'SafetyScore' (0-2)")

# 5. RISK SCORE FEATURES
print("5. RISK SCORE FEATURES")

# Combined risk score (higher = more risky)
# Based on vehicle age, power, and safety features
df_model['RiskScore_Engineered'] = (
    (df_model['VehicleAge'] / 30) * 0.3 +           # Older = higher risk
    (df_model['PowerToWeight'] * 10) * 0.4 +        # More power = higher risk
    (1 - df_model['SafetyScore'] / 2) * 0.3         # Less safety = higher risk
)
df_model['RiskScore_Engineered'] = df_model['RiskScore_Engineered'].clip(0, 1)
print("Created 'RiskScore_Engineered' (0-1, higher = more risky)")
# 6. INTERACTION FEATURES
print("6. INTERACTION FEATURES")

# Luxury vehicle indicator (BMW, Mercedes-Benz)
luxury_brands = ['BMW', 'MERCEDES-BENZ', 'AUDI', 'LEXUS']
df_model['IsLuxury'] = df_model['make'].str.upper().isin(luxury_brands).astype(int)
print("Created 'IsLuxury' (binary)")

# High performance indicator (power > 200kW)
df_model['IsHighPerformance'] = (df_model['kilowatts'] > 200).astype(int)
print("Created 'IsHighPerformance' (binary)")

#Summary
print("FEATURE ENGINEERING SUMMARY")

# Count new features created
new_features = ['VehicleAge', 'VehicleAgeCat', 'IsNewVehicle', 'PowerToWeight', 
                'EngineSizeCat', 'PowerCat', 'PremiumPerCC', 'PremiumCat', 
                'HasAlarm', 'HasTracking', 'SafetyScore', 'RiskScore_Engineered',
                'IsLuxury', 'IsHighPerformance']

existing_new = [f for f in new_features if f in df_model.columns]
print(f"\nNew features created: {len(existing_new)}")
for f in existing_new:
    print(f"  - {f}")

print(f"\nTotal columns in dataset: {df_model.shape[1]}")

1. VEHICLE AGE FEATURES
 Created 'VehicleAge' (years)
Created 'VehicleAgeCat' (New/Like New/Used/Old)
Created 'IsNewVehicle' (binary)
ENGINE PERFORMANCE FEATURES
Created 'PowerToWeight' (kW/cc)
Created 'EngineSizeCat'
Created 'PowerCat'
3. PREMIUM-BASED FEATURES
Created 'PremiumPerCC' (R per cc)
Created 'PremiumCat' (quartiles)
4. SAFETY FEATURES
Created 'HasAlarm' (binary)
Created 'HasTracking' (binary)
Created 'SafetyScore' (0-2)
5. RISK SCORE FEATURES
Created 'RiskScore_Engineered' (0-1, higher = more risky)
6. INTERACTION FEATURES
Created 'IsLuxury' (binary)
Created 'IsHighPerformance' (binary)
FEATURE ENGINEERING SUMMARY

New features created: 14
  - VehicleAge
  - VehicleAgeCat
  - IsNewVehicle
  - PowerToWeight
  - EngineSizeCat
  - PowerCat
  - PremiumPerCC
  - PremiumCat
  - HasAlarm
  - HasTracking
  - SafetyScore
  - RiskScore_Engineered
  - IsLuxury
  - IsHighPerformance

Total columns in dataset: 36


# Feature Engineering Summary

## 1. Vehicle Age Features

| Feature Name | Description | Data Type |
|--------------|-------------|-----------|
| `VehicleAge` | Vehicle age in years (capped at 30 years) | Numeric |
| `VehicleAgeCat` | Age category: New (0-2y), Like New (3-5y), Used (6-10y), Old (10+y) | Categorical |
| `IsNewVehicle` | Binary indicator for vehicles ≤2 years old | Binary (0/1) |

---

## 2. Engine Performance Features

| Feature Name | Description | Data Type |
|--------------|-------------|-----------|
| `PowerToWeight` | Power-to-weight ratio (kilowatts per cubic capacity) | Numeric |
| `EngineSizeCat` | Engine size category: Small (<1.4L), Medium (1.4-1.8L), Large (1.8-2.2L), Extra Large (>2.2L) | Categorical |
| `PowerCat` | Power category: Low (<100kW), Medium (100-200kW), High (200-300kW), Very High (>300kW) | Categorical |

---

## 3. Premium-Based Features

| Feature Name | Description | Data Type |
|--------------|-------------|-----------|
| `PremiumPerCC` | Premium amount per cubic capacity (R per cc) | Numeric |
| `PremiumCat` | Premium quartile categories | Categorical |

---

## 4. Safety Features

| Feature Name | Description | Data Type |
|--------------|-------------|-----------|
| `HasAlarm` | Binary indicator for alarm/immobiliser presence | Binary (0/1) |
| `HasTracking` | Binary indicator for tracking device presence | Binary (0/1) |
| `SafetyScore` | Combined safety score (0-2, sum of HasAlarm + HasTracking) | Numeric |

---

## 5. Risk Score Features

| Feature Name | Description | Data Type |
|--------------|-------------|-----------|
| `RiskScore_Engineered` | Combined risk score (0-1, higher = more risky) based on vehicle age, power-to-weight, and safety score | Numeric |

**Formula:**
RiskScore = (VehicleAge/30 × 0.3) + (PowerToWeight × 10 × 0.4) + ((1 - SafetyScore/2) × 0.3)

---

## 6. Interaction Features

| Feature Name | Description | Data Type |
|--------------|-------------|-----------|
| `IsLuxury` | Binary indicator for luxury brands (BMW, Mercedes-Benz, AUDI, LEXUS) | Binary (0/1) |
| `IsHighPerformance` | Binary indicator for high-performance vehicles (kilowatts > 200kW) | Binary (0/1) |

---

## Summary Statistics

| Metric | Value |
|--------|-------|
| **Total new features created** | 14 |
| **Total columns in dataset** | 36 |
| **Numeric features** | 8 |
| **Categorical features** | 6 |
| **Binary features** | 6 |

---

## Feature List

### Numeric Features
- VehicleAge
- PowerToWeight
- PremiumPerCC
- SafetyScore
- RiskScore_Engineered

### Categorical Features
- VehicleAgeCat
- EngineSizeCat
- PowerCat
- PremiumCat

### Binary Features
- IsNewVehicle
- HasAlarm
- HasTracking
- IsLuxury
- IsHighPerformance